# File I/O in Python
Covers: open() modes, read/write/readline/readlines, with statement, pathlib, binary files, csv, json, pickle

## 1. open() Modes and Basic Reading

In [ ]:
import tempfile
from pathlib import Path

# Create a temp file for demonstrations
tmpdir = tempfile.mkdtemp()
sample_file = Path(tmpdir) / 'sample.txt'
sample_file.write_text('Line 1: Hello\nLine 2: World\nLine 3: Python\n', encoding='utf-8')

# read() — entire file
with open(sample_file, 'r', encoding='utf-8') as f:
    content = f.read()
print('read():', repr(content))

# readline() — one line at a time
with open(sample_file, 'r', encoding='utf-8') as f:
    line1 = f.readline()
    line2 = f.readline()
print('readline() line1:', repr(line1))
print('readline() line2:', repr(line2))

# readlines() — all lines as list
with open(sample_file, 'r', encoding='utf-8') as f:
    lines = f.readlines()
print('readlines():', lines)

# Iterate line by line (most memory-efficient)
print('\nIterate line by line:')
with open(sample_file, 'r', encoding='utf-8') as f:
    for line in f:
        print(' ', repr(line.rstrip()))

## 2. Writing Files — write, writelines, append

In [ ]:
from pathlib import Path
import tempfile

tmpdir = Path(tempfile.mkdtemp())

# write mode — truncates existing content
output = tmpdir / 'output.txt'
with open(output, 'w', encoding='utf-8') as f:
    f.write('First line\n')
    f.write('Second line\n')
print('After write:', output.read_text())

# writelines — no auto newlines!
lines = ['Line A\n', 'Line B\n', 'Line C\n']
with open(output, 'w', encoding='utf-8') as f:
    f.writelines(lines)
print('After writelines:', output.read_text())

# append mode
with open(output, 'a', encoding='utf-8') as f:
    f.write('Appended line\n')
print('After append:', output.read_text())

# print() to file
with open(output, 'w', encoding='utf-8') as f:
    print('Hello', 'World', sep=', ', file=f)
    print('Second line', file=f)
print('After print():', output.read_text())

# Exclusive creation — fails if file exists
new_file = tmpdir / 'new.txt'
try:
    with open(new_file, 'x') as f:
        f.write('Created exclusively')
    print('Created new file')
    with open(new_file, 'x') as f:  # second attempt fails
        pass
except FileExistsError as e:
    print('FileExistsError:', e)

## 3. seek() and tell() — File Position

In [ ]:
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())
f_path = tmpdir / 'seek_demo.txt'
f_path.write_bytes(b'Hello, World! Python is great.')

with open(f_path, 'rb') as f:
    print('Initial position:', f.tell())  # 0
    
    data = f.read(5)
    print('After read(5):', data, '| position:', f.tell())
    
    f.seek(0)  # back to start
    print('After seek(0):', f.tell())
    
    f.seek(0, 2)  # seek to end
    size = f.tell()
    print('File size:', size)
    
    f.seek(-6, 2)  # 6 bytes from end
    tail = f.read()
    print('Last 6 bytes:', tail)

## 4. pathlib.Path — Modern File I/O

In [ ]:
from pathlib import Path
import tempfile

tmpdir = Path(tempfile.mkdtemp())

# Text operations
txt = tmpdir / 'hello.txt'
txt.write_text('Hello, pathlib!\nSecond line.', encoding='utf-8')
content = txt.read_text(encoding='utf-8')
print('read_text:', content)

# Binary operations
bin_file = tmpdir / 'data.bin'
bin_file.write_bytes(bytes(range(16)))
data = bin_file.read_bytes()
print('read_bytes:', data.hex())

# Path properties
p = Path('/home/user/documents/report.pdf')
print('\nPath properties:')
print('  name:', p.name)
print('  stem:', p.stem)
print('  suffix:', p.suffix)
print('  parent:', p.parent)
print('  parts:', p.parts)

# Glob
for i in range(3):
    (tmpdir / f'file{i}.py').write_text(f'# file {i}')
(tmpdir / 'data.txt').write_text('data')

py_files = sorted(tmpdir.glob('*.py'))
print('\nPython files:', [f.name for f in py_files])

all_files = sorted(tmpdir.rglob('*'))
print('All files:', [f.name for f in all_files if f.is_file()])

## 5. Binary Files

In [ ]:
import tempfile
from pathlib import Path
import struct

tmpdir = Path(tempfile.mkdtemp())

# Write binary data
bin_file = tmpdir / 'data.bin'
with open(bin_file, 'wb') as f:
    # Write structured binary data
    f.write(b'MAGIC')           # 5-byte magic number
    f.write(struct.pack('>I', 42))  # 4-byte big-endian uint32
    f.write(struct.pack('>f', 3.14))  # 4-byte float
    f.write(bytes([0, 1, 2, 3, 255]))  # raw bytes

# Read binary data
with open(bin_file, 'rb') as f:
    magic = f.read(5)
    number = struct.unpack('>I', f.read(4))[0]
    pi_approx = struct.unpack('>f', f.read(4))[0]
    raw = f.read(5)

print('Magic:', magic)
print('Number:', number)
print('Float:', round(pi_approx, 4))
print('Raw bytes:', list(raw))

# Copy file in chunks (memory-efficient)
source = tmpdir / 'source.bin'
dest = tmpdir / 'dest.bin'
source.write_bytes(bytes(range(256)) * 100)  # 25.6KB

with open(source, 'rb') as src, open(dest, 'wb') as dst:
    while chunk := src.read(1024):
        dst.write(chunk)

print('\nCopied file size:', dest.stat().st_size, 'bytes')

## 6. csv Module

In [ ]:
import csv
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())
csv_file = tmpdir / 'people.csv'

# DictWriter — write with column names
people = [
    {'name': 'Alice', 'age': 30, 'city': 'New York', 'salary': 95000},
    {'name': 'Bob', 'age': 25, 'city': 'London', 'salary': 72000},
    {'name': 'Charlie', 'age': 35, 'city': 'Tokyo', 'salary': 88000},
]

with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['name', 'age', 'city', 'salary']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(people)

print('Written CSV:')
print(csv_file.read_text())

# DictReader — read as dicts
print('Read back:')
with open(csv_file, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['name']}, age {row['age']}, salary ${row['salary']}")

# csv.reader — basic
print('\ncsv.reader:')
with open(csv_file, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    print('Header:', header)
    for row in reader:
        print(' ', row)

## 7. json and pickle

In [ ]:
import json
import pickle
import tempfile
from pathlib import Path
from datetime import datetime

tmpdir = Path(tempfile.mkdtemp())

# JSON
data = {
    'users': [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}],
    'count': 2,
    'active': True
}

json_file = tmpdir / 'data.json'
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2)

with open(json_file, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('JSON round-trip:', loaded['users'][0]['name'])

# Custom JSON serializer
def json_default(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    if isinstance(obj, set):
        return sorted(list(obj))
    raise TypeError(f'Not serializable: {type(obj).__name__}')

special_data = {'ts': datetime.now(), 'tags': {'python', 'json'}}
json_str = json.dumps(special_data, default=json_default, indent=2)
print('\nCustom JSON:', json_str)

# Pickle
class Model:
    def __init__(self, weights):
        self.weights = weights
        self._cache = {}  # won't be pickled
    
    def __getstate__(self):
        state = self.__dict__.copy()
        del state['_cache']
        return state
    
    def __setstate__(self, state):
        self.__dict__.update(state)
        self._cache = {}
    
    def predict(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x))

model = Model([0.5, 1.2, -0.3])
pkl_file = tmpdir / 'model.pkl'

with open(pkl_file, 'wb') as f:
    pickle.dump(model, f)

with open(pkl_file, 'rb') as f:
    loaded_model = pickle.load(f)

print('\nPickle round-trip prediction:', loaded_model.predict([1, 2, 3]))
print('Cache reinitialized:', loaded_model._cache)